# tAilorCode · Prueba guiada de una tarea

**Autora: Araceli Fradejas Muñoz.** Cuaderno limpio preparado el 16 de octubre de 2026.

Objetivo: preparar Gemma, darle una incidencia pública de FastAPI y conservar el resultado.
Este cuaderno no envía nada a Kaggle. No conserva celdas ni errores antiguos.

**Hito comprobado el 16 de octubre de 2026:** `fastapi_14786` resuelta, `test_exit_code: 0`, 6 llamadas registradas y 43,65 segundos de evaluación. Es un registro del intento terminado, no una ejecución de esta copia limpia. Los diagnósticos se guardan por separado.

Ejecuta un bloque cada vez. **No uses «Ejecutar todas».** Antes de conectar, acuerda duración y consumo.
La tasa observada el 16 de octubre fue de 5,3 unidades/hora en A100; puede cambiar.
Guardar el cuaderno no conserva la máquina ni sus archivos temporales.

Ruta: autorización → máquina → acceso Kaggle → proyecto → instalación → datos/modelo → comprobaciones → tarea → guardar y desconectar.

## 1. Autorizar la sesión

Selecciona el entorno candidato Colab 2026.04 / Python 3.12 y una A100 cuando se acuerde el consumo.
Escribe un plazo absoluto con zona horaria, por ejemplo en formato `AAAA-MM-DDTHH:MM:SS+00:00` para UTC.
Cambiar este plazo requiere un nuevo acuerdo. El control bloquea pasos nuevos, pero **no interrumpe procesos ni detiene la facturación**.

In [ ]:
RUNTIME_COST_APPROVED = False
SESSION_END_UTC = ""
from datetime import datetime, timezone
if not RUNTIME_COST_APPROVED:
    raise RuntimeError("Pendiente de acordar duración y consumo de esta sesión.")
session_end = datetime.fromisoformat(SESSION_END_UTC)
if session_end.utcoffset() is None:
    raise RuntimeError("El plazo necesita zona horaria.")
def require_session_time():
    if not RUNTIME_COST_APPROVED or datetime.now(timezone.utc) >= session_end:
        raise RuntimeError("Sesión no autorizada o plazo terminado. Guardar resultados y desconectar.")
require_session_time()
print("Plazo UTC:", session_end.isoformat())

## 2. Comprobar la máquina

Comprueba Python, espacio de disco y GPU. Esto no carga el modelo.
Una A100 de 40 GB se ha detectado previamente; que aparezca la GPU no garantiza que toda la ejecución funcione.

In [ ]:
if not globals().get("RUNTIME_COST_APPROVED", False):
    raise RuntimeError("Runtime approval is required before this step.")

require_session_time()

import sys
import platform
import shutil
from importlib import metadata

print("Python:", sys.version.split()[0])
print("Sistema:", platform.system(), platform.machine())
print("Disco libre (GiB):",
      round(shutil.disk_usage("/content").free / 1024**3, 1))

for name in ("kagglehub", "kaggle", "torch", "transformers", "vllm"):
    try:
        version = metadata.version(name)
    except metadata.PackageNotFoundError:
        version = "no instalado"
    print(f"{name}: {version}")

import subprocess
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No NVIDIA GPU detected; stop before GPU installation.")
gpu = subprocess.run([
    "nvidia-smi", "--query-gpu=name,memory.total,driver_version",
    "--format=csv,noheader",
], text=True, capture_output=True, timeout=20)
print(gpu.stdout)
print(gpu.stderr)
gpu.check_returncode()

## 3. Acceso temporal a Kaggle

Usa el secreto existente `KAGGLE_API_TOKEN` en el panel de la llave y permite acceso temporal a este cuaderno.
No pegues su valor en código ni salidas. Cada persona necesita su propia cuenta y acceso autorizado a la competición.
Se revoca este acceso antes de ejecutar código de la tarea. No montes Drive ni archivos personales en la máquina de evaluación.

In [ ]:
if not globals().get("RUNTIME_COST_APPROVED", False):
    raise RuntimeError("Runtime approval is required before this step.")

require_session_time()

import kagglehub

kagglehub.whoami(verbose=False)
print("Autenticación de Kaggle verificada.")

## 4. Preparar el proyecto

Descarga una revisión fijada del proyecto y tres paquetes oficiales pequeños. Se verifican sus hashes.
Usa una máquina nueva; si ya existen estas carpetas, revisa su contenido antes de repetir.

In [ ]:
if not globals().get("RUNTIME_COST_APPROVED", False):
    raise RuntimeError("Runtime approval is required before this step.")

require_session_time()

import hashlib
import json
import platform
import shutil
import subprocess
import sys
from pathlib import Path
import kagglehub

if (platform.system(), platform.machine(), sys.version_info[:2]) != ("Linux", "x86_64", (3, 12)):
    raise RuntimeError("Use Linux x86_64 and Python 3.12.")
REPO = Path("/content/tailorcode-setup-20261004")
REVISION = "8fd6267741b786bce69bb150182ba264a3b73ffe"
WHEELS = Path("/content/tailorcode-harness-wheels")
VENV = Path("/content/tailorcode-gpu-venv-02")
PYTHON = VENV / "bin/python"
if REPO.exists():
    raise RuntimeError("Project folder exists. Inspect it before retrying; do not overwrite.")
subprocess.run(["git", "clone", "https://github.com/AraceliFradejas/tailorcode-gemma-developer-agent.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", REVISION], check=True)
print("Project revision:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())
WHEELS.mkdir(exist_ok=True)
manifest = json.loads((REPO / "evaluation/official-harness-wheels.json").read_text())
for entry in manifest["files"]:
    downloaded = Path(kagglehub.dataset_download("metric/gemma-4-developer-agent-wheelhouse/versions/25", path=entry["name"]))
    if hashlib.sha256(downloaded.read_bytes()).hexdigest() != entry["sha256"]:
        raise RuntimeError(f"Official wheel hash mismatch: {entry['name']}")
    shutil.copy2(downloaded, WHEELS / entry["name"])
    print("Verified:", entry["name"])

## 5. Instalar las librerías

La instalación descarga varios GB y puede tardar. Crea un entorno Python separado y conserva el registro.
Se fija FastAPI 0.136.3 para evitar el error `_IncludedRouter` observado en el servidor.
Activa este bloque solo en la sesión autorizada. No continúes si termina con error.

In [ ]:
INSTALL_APPROVED = False
if not globals().get("RUNTIME_COST_APPROVED", False) or not INSTALL_APPROVED:
    raise RuntimeError("Installation disabled; approve this preparation step separately.")

require_session_time()

import importlib.util
import subprocess
import sys

requirements = REPO / "requirements-gpu.txt"
requirements_text = requirements.read_text()
if "fastapi==0.136.3" not in requirements_text:
    requirements.write_text(requirements_text + "\nfastapi==0.136.3\n")

if VENV.exists():
    raise RuntimeError("Environment path already exists; preserve it and choose a new path.")
if importlib.util.find_spec("ensurepip") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements-setup.txt")], check=True, timeout=120)
install_log = REPO.parent / "tailorcode-install-02.log"
command = [
    sys.executable, "-u", str(REPO / "scripts/install_gpu_environment.py"),
    "--wheelhouse", str(WHEELS), "--venv", str(VENV), "--install",
]
with install_log.open("x", encoding="utf-8") as log:
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end="", flush=True)
            return_code = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
print("Installation exit code:", return_code)
if return_code != 0:
    raise RuntimeError("Installation failed. Preserve the log; do not start Gemma.")
print("Environment ready:", PYTHON)
print("No model started.")

## 6. Comprobar CUDA y el componente HTTP

CUDA permite que PyTorch utilice la GPU. La segunda prueba comprueba salud y métricas sin cargar Gemma.
Esta comprobación detecta el fallo HTTP anterior, pero no sustituye el arranque completo del servidor.

In [ ]:
require_session_time()
subprocess.run([str(PYTHON), "-c", "import torch; print('CUDA:', torch.cuda.is_available()); assert torch.cuda.is_available()"], check=True, timeout=60)
http_check = '"""Exercise the server\'s metrics/HTTP integration on CPU without loading a model."""\nfrom importlib.metadata import version\n\n\ndef main():\n    from fastapi import APIRouter, FastAPI\n    from fastapi.testclient import TestClient\n    from prometheus_fastapi_instrumentator import Instrumentator\n\n    app = FastAPI()\n    router = APIRouter()\n\n    @router.get(\'/health\')\n    def health():\n        return {\'status\': \'ok\'}\n\n    app.include_router(router)\n    Instrumentator().instrument(app).expose(app)\n    with TestClient(app) as client:\n        for path in (\'/health\', \'/metrics\'):\n            response = client.get(path)\n            if response.status_code != 200:\n                raise RuntimeError(f\'{path}: HTTP {response.status_code}\')\n        if client.get(\'/health\').json() != {\'status\': \'ok\'}:\n            raise RuntimeError(\'Unexpected health response\')\n    for package in (\'fastapi\', \'starlette\', \'prometheus-fastapi-instrumentator\'):\n        print(f\'{package}: {version(package)}\')\n    print(\'HTTP and metrics checks passed on CPU. No model loaded.\')\n\n\nif __name__ == \'__main__\':\n    main()\n'
subprocess.run([str(PYTHON), "-c", http_check], check=True, timeout=60)

## 7. Preparar la tarea

Descarga unos 276 MiB: una copia del repositorio, metadatos y dependencias para ejecutar sus pruebas sin Internet.
Verifica 129 archivos mediante hashes. No ejecuta la tarea ni muestra sus parches de referencia.

In [ ]:
DOWNLOAD_TASK_INPUTS = False
if not globals().get("RUNTIME_COST_APPROVED", False) or not DOWNLOAD_TASK_INPUTS:
    raise RuntimeError("Task downloads disabled. Approve runtime and this download first.")

require_session_time()

import json
import shutil
import sys
from pathlib import Path
import kagglehub

if not (REPO / "scripts/prepare_colab_inputs.py").is_file():
    raise RuntimeError("The prepared project is outdated; use the notebook's pinned revision.")
sys.path.insert(0, str(REPO / "scripts"))
from prepare_colab_inputs import prepare_task

data_directory = Path("/content/tailorcode-data")
data_directory.mkdir(parents=True, exist_ok=True)
manifest = json.loads((REPO / "evaluation/colab-inputs.json").read_text())
needed = sum(entry["bytes"] for entry in manifest["files"])
if shutil.disk_usage(data_directory).free < 2 * needed + 5 * 1024**3:
    raise RuntimeError("Insufficient disk headroom. No task downloads started.")
report = prepare_task(manifest, data_directory, kagglehub.competition_download)
print(json.dumps(report, indent=2))
print("No repository code or model was executed.")

## 8. Preparar Gemma

Solicita la versión fijada del modelo, de unos 23,3 GB. Si Colab la hace accesible mediante una ruta montada,
este paso puede ser rápido. Comprobar archivos presentes no prueba la integridad numérica ni el funcionamiento en GPU.
Los archivos temporales pueden desaparecer al eliminar la máquina.

In [ ]:
DOWNLOAD_MODEL_WEIGHTS = False
if not globals().get("RUNTIME_COST_APPROVED", False) or not DOWNLOAD_MODEL_WEIGHTS:
    raise RuntimeError("Model download disabled. Approve this large download separately.")

require_session_time()

import json
import shutil
import sys
from pathlib import Path
import kagglehub

if not (REPO / "scripts/prepare_colab_inputs.py").is_file():
    raise RuntimeError("The prepared project is outdated; use the notebook's pinned revision.")
data_directory = Path("/content/tailorcode-data")
data_directory.mkdir(parents=True, exist_ok=True)
if shutil.disk_usage(data_directory).free < 55 * 1024**3:
    raise RuntimeError("Insufficient disk headroom. No model download started.")
sys.path.insert(0, str(REPO / "scripts"))
from prepare_colab_inputs import MODEL, validate_model

print("Downloading full pinned model (~23.3 GB). No inference.", flush=True)
model_directory = Path(kagglehub.model_download(MODEL))
report = validate_model(model_directory)
(data_directory / "model-check.json").write_text(json.dumps(report, indent=2) + "\n")
print(json.dumps(report, indent=2))

## 9. Revisar requisitos

Configura rutas reales y genera una carpeta nueva para el resultado. El preflight debe indicar `blockers: []`.
Pasar este control no significa haber resuelto la tarea.

In [ ]:
if not globals().get("RUNTIME_COST_APPROVED", False):
    raise RuntimeError("Runtime approval is required")

require_session_time()

from pathlib import Path
from datetime import datetime, timezone
import json
import subprocess

REPO = Path("/content/tailorcode-setup-20261004")
PYTHON = Path("/content/tailorcode-gpu-venv-02/bin/python")
if not (REPO / "scripts/run_public_smoke.py").is_file() or not PYTHON.is_file():
    raise RuntimeError("Project or isolated interpreter is missing; prepare this runtime first.")
DATA_ROOT = "/content/tailorcode-data"
model_report = Path(DATA_ROOT) / "model-check.json"
if not (Path(DATA_ROOT) / "input-check.json").is_file() or not model_report.is_file():
    raise RuntimeError("Verified task/model preparation reports are missing.")
MODEL_PATH = json.loads(model_report.read_text())["model_path"]
TASK_ID = "fastapi_14786"
OUTPUT = Path("/content/tailorcode-runs") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
print("Evaluation interpreter:", PYTHON)

In [ ]:
if not globals().get("RUNTIME_COST_APPROVED", False):
    raise RuntimeError("Runtime approval is required before this step.")

require_session_time()

if not RUNTIME_COST_APPROVED or not DATA_ROOT or not MODEL_PATH:
    raise RuntimeError("Runtime approval and real DATA_ROOT / MODEL_PATH paths are required")
check = subprocess.run([
    str(PYTHON), str(REPO / "scripts/check_smoke_environment.py"),
    "--data-root", DATA_ROOT, "--model-path", MODEL_PATH, "--task-id", TASK_ID,
], text=True, capture_output=True)
print(check.stdout)
print(check.stderr)
check.check_returncode()

## 10. Lanzar una tarea

Desactiva el acceso a `KAGGLE_API_TOKEN` en Secretos y comprueba visualmente que queda apagado.
Después activa los dos indicadores de esta celda para la prueba autorizada.
El agente dispone de cinco minutos; carga del servidor y verificación necesitan tiempo adicional.
La máquina continúa consumiendo unidades hasta que se termina su entorno.

In [ ]:
require_session_time()
RUN_AGENT = False
SECRET_ACCESS_REVOKED = False
if not RUN_AGENT or not SECRET_ACCESS_REVOKED:
    raise RuntimeError("Falta aprobar la prueba o revocar el acceso al secreto.")
import os, subprocess
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
log_path = OUTPUT.parent / (OUTPUT.name + ".log")
clean_env = {key: value for key, value in os.environ.items() if not any(word in key.upper() for word in ("TOKEN", "SECRET", "PASSWORD", "API_KEY", "CREDENTIAL"))}
with log_path.open("x") as log:
    run_process = subprocess.Popen([str(PYTHON), "-u", str(REPO / "scripts/run_public_smoke.py"), "--data-root", DATA_ROOT, "--model-path", MODEL_PATH, "--task-id", TASK_ID, "--output", str(OUTPUT), "--run-agent"], stdout=log, stderr=subprocess.STDOUT, env=clean_env, start_new_session=True)
print("Proceso:", run_process.pid)
print("Resultados:", OUTPUT)
print("Registro:", log_path)

## 11. Seguir el progreso y entender el resultado

Puedes repetir esta celda de lectura sin reiniciar la prueba.
`starting` significa arranque; `evaluating`, evaluación en marcha; `completed`, evaluación terminada.
`resolved: true` indica que el parche pasó la verificación de esta tarea. No es una puntuación de Kaggle.
Si aparece `failed`, lee `error` y el registro antes de decidir el siguiente paso.

In [ ]:
import json
if "run_process" in globals():
    print("Estado del proceso:", run_process.poll())
if "OUTPUT" in globals():
    report_file = OUTPUT / "run.json"
    if report_file.is_file():
        current = json.loads(report_file.read_text())
        print(json.dumps({key: current[key] for key in ("status", "inference", "resolved", "error", "test_exit_code", "tool_calls", "duration_seconds") if key in current}, indent=2))
    else:
        print("Todavía no hay informe.")
if "log_path" in globals() and log_path.is_file():
    print(log_path.read_text()[-3000:])

## 12. Guardar y terminar

Descarga los diagnósticos antes de eliminar la máquina. El ZIP incluye registros, versiones y el parche generado cuando exista;
excluye pesos y archivos de tareas. Revisa los registros antes de publicarlos.
Esta celda de guardado funciona aunque el plazo haya vencido; no inicia el modelo.

Después: **Entorno de ejecución → Desconectar y eliminar entorno de ejecución**.
En Recursos o Gestionar sesiones comprueba que quedan **0 sesiones activas**. Guarda también el cuaderno.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
paths = [Path("/content/tailorcode-install-02.log")]
if "VENV" in globals():
    paths.append(VENV / "installed-packages.txt")
if "OUTPUT" in globals():
    paths.extend([OUTPUT / "run.json", OUTPUT / "agent.patch", OUTPUT.parent / (OUTPUT.name + ".log")])
available = [path for path in paths if path.is_file()]
if not available:
    raise RuntimeError("Todavía no hay diagnósticos; conserva las salidas de error del cuaderno.")
archive = Path("/content") / ("tailorcode-diagnostics-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".zip")
with ZipFile(archive, "x", compression=ZIP_DEFLATED) as bundle:
    for path in available:
        bundle.write(path, arcname=path.name)
print("Incluidos:", [path.name for path in available])
files.download(str(archive))